# <b> Data Engineering in Databricks: Data Ingestion with LakeFlow Connect </b>

## 1. Databricks LakeFlow Platform Overview
LakeFlow provides an end-to-end data engineering platform to replace fragmented traditional ingestion pipelines (which suffer from high overhead, inconsistent governance, and poor observability).

### Core Components
* **LakeFlow Connect:** Scalable, managed ingestion connectors for enterprise apps, databases, cloud storage, message buses, and local files.
* **Spark Declarative Pipelines (SDP):** A framework for building batch and streaming data transformation pipelines using SQL and Python.
* **LakeFlow Jobs:** Automated workflow orchestration to schedule and coordinate multi-task processing pipelines.

## 2. LakeFlow Connectors

| Connector Category | Supported Sources | Ingestion Patterns & Features |
| :--- | :--- | :--- |
| **Upload Files** | Local desktop files | Direct manual upload to Unity Catalog volumes or tables. |
| **Standard Connectors** | Cloud Object Storage, Apache Kafka, etc. | Supports Batch, Incremental Batch, and Streaming modes. |
| **Managed Connectors** | Databases, SaaS Applications | Uses efficient incremental reads/writes for scalable ingestion. |

## 3. Data Ingestion Methods

* **Batch Ingestion:** Processes complete datasets in single execution runs (e.g., using `CREATE TABLE AS SELECT` or `COPY INTO`). Ideal for historical data loading where latency is not critical.
* **Incremental Batch:** Automatically detects and processes only new or updated records while skipping previously ingested data to optimize compute costs.
* **Streaming Ingestion:** Continuously loads rows or micro-batches as data arrives in near real-time. Common APIs include `spark.readStream` (using Auto Loader) and Declarative Pipelines with continuous triggers. Supports sources such as Apache Kafka, Amazon Kinesis, Google Pub/Sub, and Apache Pulsar.

## 4. Delta Lake & Unity Catalog (UC) Tables
Delta Lake is an open-source storage protocol that layers JSON-based transaction logs (`_delta_log/`) over standard Parquet data files to maintain table state.

### Key Features
* **ACID Transactions:** Ensures atomicity, consistency, isolation, and durability for concurrent multi-user operations.
* **DML Support:** Full support for `INSERT`, `UPDATE`, `DELETE`, and `MERGE INTO` operations.
* **Time Travel:** Enables querying historical snapshots or reverting table states for auditing and recovery.
* **Schema Governance:** Enforces strict data quality constraints while allowing controlled schema evolution.

## 5. Medallion Architecture

* **Bronze (Raw Data):** Stores raw, unmodified source data "as is" to serve as an immutable historical record.
* **Silver (Cleaned Data):** Contains cleansed, filtered, and enriched data optimized for general analysis.
* **Gold (Curated Data):** High-level business aggregations structured for BI reporting, dashboards, and AI/ML applications.

This reference document summarizes key concepts from the [Data Engineering in Databricks](https://customer-academy.databricks.com/learn/learning-plans/10/data-engineer-learning-plan/courses/2963/data-ingestion-with-lakeflow-connect/lessons/62746:4036/data-engineering-in-databricks) lesson for study and repository documentation.

---

# <b> Data Ingestion from Cloud Storage </b>
Raw files in cloud object storage (CSV, JSON, Parquet, etc.) are converted into Delta tables, unlocking ACID transactions, time travel, and schema enforcement. Ingestion from cloud storage uses LakeFlow Connect Standard Connectors.
## Ingestion Methods Overview
- **CREATE TABLE AS (CTAS):** Batch ingestion that creates a Delta table by default from files using `read_files()`. Best for smaller, ad hoc datasets.
- **COPY INTO:** Incremental batch ingestion that is idempotent and retriable. Skips files already loaded and supports format and copy options for fine-grained control. Ideal for thousands of files.
- **Auto Loader:** The most scalable method, built on Spark Structured Streaming. Supports incremental batch or streaming, in both Python and SQL (via Declarative Pipelines), processes billions of files, and automatically handles schema evolution.

## 1. CREATE TABLE AS (CTAS)
- **Ingestion Type:** Batch. Creates a Delta table by default from files in cloud object storage.
- **Syntax:**
```
    CREATE TABLE new_table AS
        SELECT 
            *
        FROM read_files(
            <path_to_file(s)>,
            format => '<file_type>',
            <other_format_specific_options>
        );
```
- **read_files():** Reads files under a provided location and returns the data in tabular form.
- **Supported Formats:** JSON, CSV, XML, TEXT, BINARYFILE, PARQUET, AVRO, and ORC.
- **Schema:** Can detect the file format automatically and infer a unified schema across all files. Format-specific options can be set based on the source file format.
- **Idempotency:** Not idempotent, so all data is read again on each run. Best for one-time, ad hoc ingestion of smaller datasets.
- **Streaming:** The same pattern can be used in streaming tables to ingest files incrementally into Delta Lake using Auto Loader.

## 2. COPY INTO
- **Ingestion Type:** Incremental batch. Performs a bulk load of files from cloud object storage into an existing Delta table (create the table first, for example with `CREATE TABLE new_table`).
- **Syntax:** 
```
    CREATE TABLE new_table;
            
        COPY INTO new_table
        FROM '<dir_path>'
        FILEFORMAT = <file_type>
        FORMAT_OPTIONS (<options>)
        COPY_OPTIONS (<options>)
```
- **Idempotent:** Skips any files that have already been loaded into the table, so only new files are ingested. Ideal when the storage location continuously receives files.
- **FROM Clause:** Specifies the path of the cloud storage location where new files are being added.
- **FORMAT_OPTIONS:** Controls how source files are parsed and interpreted (options depend on the file format).
- **COPY_OPTIONS:** Controls the behavior of the `COPY INTO` operation itself, such as schema evolution with `mergeSchema` and idempotency with `force`.
- **File Format Support:** Parquet, JSON, XML, and others.

## 3. Auto Loader
- **Ingestion Type:** Incremental batch or streaming. Processes new data files incrementally as they arrive in cloud storage, with no extra setup or complex configuration.
- **Capabilities:** Automatically detects and loads new files into Delta tables, works with both Python and SQL (via Declarative Pipelines), scales to billions of files, and relies on Spark Structured Streaming for efficient and reliable ingestion.
- **Spark Auto Loader:** Start with `spark.readStream` and set the format to `cloudFiles`, which enables Auto Loader. Set `cloudFiles.format` (for example JSON) and `cloudFiles.schemaLocation` to track schema inference and evolution. Use `.load()` to point to the files (for example a Unity Catalog path under `/Volumes`).
**Write Side:** Use `.writeStream` with a `checkpointLocation` to maintain state and progress, a `.trigger(processingTime="5 seconds")` interval, and `.toTable()` to write into a Delta table (catalog, schema, table).
```
    (spark
    .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "<checkpoint_path>")
        .load("/Volumes/catalog/schema/files")
    .writeStream
        .option("checkpointLocation", "<checkpoint_path>")
        .trigger(processingTime="5 seconds")
        .toTable("catalog.database.table")
    )
```
- **Auto Loader with SQL (Declarative Pipelines):** 
```
    CREATE OR REFRESH STREAMING TABLE
        catalog.schema.table
    SCHEDULE EVERY 1 HOUR
    AS
    SELECT 
        *
    FROM STREAM read_files(
        '<dir_path>',
        format => '<file_type>'
    )
```

## Ingestion Methods at a Glance

| Feature | CTAS + spark.read | COPY INTO | Auto Loader |
|---|---|---|---|
| Ingestion Type | Batch | Incremental Batch | Incremental (Batch or Streaming) |
| Use Cases | Smaller datasets, one-time or ad hoc loads | Thousands of files | Millions+ files per hour, backfills with billions of files |
| Interface | Python `spark.read`, SQL CTAS | SQL | Python `spark.readStream`, SQL streaming tables |
| Idempotent | No | Yes | Yes |
| Schema Evolution | Manual or inferred during read | Supported with options (`mergeSchema`) | Automatic, handles new columns as they appear |
| Latency | High | Moderate (scheduled) | Low or high depending on configuration |
| Ease of Use | Simple | Simple and SQL-based | Intermediate to advanced |

## 6. Databricks SQL Recommendations
- **Streaming Tables:** Databricks recommends streaming tables instead of `COPY INTO` to ingest data with Databricks SQL. A streaming table is registered in Unity Catalog and a pipeline is automatically generated when it is created.
- **Sources:** Streaming tables support incremental loading from both Kafka and cloud object storage.
- **Auto Loader + Declarative Pipelines:** Recommended for most ingestion tasks from cloud object storage. Together they incrementally and idempotently load continuously growing datasets as they arrive.
- **Serverless Requirement:** Streaming tables in Databricks SQL are backed by serverless Spark Declarative Pipelines, so the workspace must support serverless pipelines. Alternatively, build your own Spark Declarative Pipelines.

## 7. Exam Tips
- **Choosing a Method:** One-time or small data, use CTAS. Thousands of files with simple scheduled SQL, use COPY INTO. Huge scale, streaming, or automatic schema evolution, use Auto Loader.
- **Idempotency:** CTAS is not idempotent. COPY INTO and Auto Loader are idempotent.
- **COPY_OPTIONS:** `mergeSchema` is for schema evolution and `force` is for idempotency.
- **Auto Loader Locations:** `cloudFiles.schemaLocation` tracks schema inference and evolution, while `checkpointLocation` maintains stream state and progress.

---

# <b> Appending Metadata Columns on Ingest </b>
Metadata columns such as the source file name and file modification time can be appended during ingestion from cloud storage using the `_metadata` column, preserving essential context for each row during table creation in the Lakehouse.

## 1. Why Metadata Columns
- **Scenario:** A set of raw files (CSV, TXT, JSON, or other formats) in cloud storage is ingested into a bronze-level table.
- **Added Columns:** Extra columns are added to each row with information from the ingestion source, such as the last modification time of the file the row came from and the source file name.
- **Purpose:** Preserves context about the data's origin, which is valuable for auditing, lineage, and debugging.
- **Result:** The Bronze table contains the original columns plus the metadata columns (for example `last_mod_time` and `source`).

## 2. The _metadata Column
- **Hidden Column:** `_metadata` is a special hidden column available for all input file formats.
- **Explicit Selection:** To include it in the returned DataFrame, you must explicitly select it in your read query when specifying the source.
- **Content:** It contains a variety of useful fields, and two are commonly used (see below).

## 3. Common _metadata Fields

| Field | Returns | Use |
|---|---|---|
| `_metadata.file_modification_time` | Last modification timestamp of the input file | Add the last file modification time |
| `_metadata.file_name` | Name of the source file for each row | Add the input (source) file name |


---

# <b> Working with the Rescued Data Column </b>
The rescued data column (`_rescued_data`) captures mismatched or unparseable fields as JSON during data ingestion, preserving non-conforming input values in your Lakehouse tables instead of dropping them.

## 1. Purpose of the Rescued Data Column
- **Problem:** During data ingestion there are times when the input data does not match the schema of your table.
- **Rescue Instead of Drop:** Columns that do not match the schema are rescued instead of being dropped.
- **Storage Format:** Mismatched values are stored as JSON-formatted strings in the `_rescued_data` column.
- **Clean Rows:** If a row has no schema mismatches, `_rescued_data` is `null` for that row.
- **Benefit:** Preserves all input data and prevents silent data loss, so you can inspect and address schema mismatches after ingestion.

## 2. Where It Is Available
- **Supported Methods:** The rescued data column is provided when ingesting with `read_files()` (CTAS), `spark.read`, or Auto Loader whenever the raw data does not match the schema.
- **COPY INTO:** Not listed among these methods. It is shown greyed out in the lecture diagram.
- **Sources:** Raw files in cloud storage such as CSV, JSON, Parquet, and other formats are ingested into a Bronze table.

## 3. Example: Rescued Data in Action
- **Setup:** Raw files contain a `users` column and a `cost` column. When ingested, `users` must be read as a STRING and `cost` must be read as a BIGINT.
- **Row 1 (Peter, $100):** `Peter` is read correctly into `users`. The value `$100` is a string that does not match BIGINT, so it is not inserted into `cost` (it is `null`). It is captured in `_rescued_data` as a JSON-formatted string.
- **Row 2 (zebi, 300):** Both values are valid (STRING and BIGINT), so the row is read into the Bronze table without issues and `_rescued_data` stays `null`.

| users | cost | _rescued_data |
|---|---|---|
| Peter | null | `{"cost": "$100", "_file_path": "<file_path>"}` |
| zebi | 300 | null |

## 4. Interpreting _rescued_data Values
- **Format:** A JSON string holding the mismatched column and its original value, plus `_file_path` with the path of the source file the row came from.
- **Non-null Value:** The row had at least one value that did not match the expected schema, and that value was rescued instead of being lost.
- **Null Value:** The row matched the schema, so nothing needed to be rescued.
- **Affected Column:** The mismatched column itself is `null` in the table, since the value did not fit the expected type.


---

# <b> Ingesting Semi-Structured Data: JSON </b>
Ingesting semi-structured data such as JSON enables efficient parsing and transformation of complex, nested input into structured Delta tables for advanced analytics in the Lakehouse.

## 1. JSON Structure
- **Objects:** JSON data is made up of JSON objects, which are typically enclosed in curly brackets `{}`.
- **Keys:** Within the curly brackets, objects contain key-value pairs. Each key is always a string enclosed in quotation marks, and each key contains a value.
- **Values:** The value of a key can be one of:
  - String
  - Number
  - Boolean
  - Array
  - Object (nested)
  - Null
- **Flat or Nested:** Objects can be flat or nested, and the complexity depends on how the data is structured. Understanding this format is important for parsing and transformation.

## 2. JSON-Formatted Columns
- **Scenario:** After ingestion, one or more table columns may contain JSON-formatted strings as values. This is common when JSON is not fully parsed during ingestion, or when JSON is embedded within another field, like a log message or a nested structure.
- **Goal:** Parse, extract, and manipulate those JSON strings using SQL or DataFrame operations, so nested fields can be flattened or accessed like regular columns.

## 3. Three Approaches for JSON Columns

| Approach | How It Works | Trade-offs |
|---|---|---|
| STRING | JSON stored as raw text. Subfields are accessed with the colon (`:`) syntax. | Holds any JSON without constraints, but is less performant than typed approaches. |
| STRUCT | JSON is parsed into a STRUCT by defining a schema. | Enforces the schema (consistent types and structure) and is more efficient to query than a raw STRING. |
| VARIANT | Newest approach for JSON in Databricks. Can store any type of data, including JSON. | Highly flexible (no rigid schema) with better performance than STRING and STRUCT. Public preview as of 2025 Q2, with General Availability (GA) still to come. |

## 4. STRING Approach: Colon Syntax
- **Access Subfields:** Use the colon (`:`) syntax to extract fields directly from the JSON string stored in a column. This syntax allows you to extract specific fields without parsing the whole column.
- **Examples:**
  - `json_column:name` returns the `name` value (John Doe).
  - `json_column:address:city` returns the nested `city` value (Anytown).

## 5. STRUCT Approach: Mapping JSON to a STRUCT
- **Define the Schema:** Tells Databricks how to interpret each part of the JSON string and convert it into the appropriate data types within a STRUCT.
- **STRUCT<> Container:** Holds the entire structure of the JSON string, preserving data types and hierarchy within a single column.
- **Map Each Key:** Go through the key-value pairs one by one and define the type of each field. In the lecture example:
  - `name`: STRING
  - `age`: INT
  - `address`: nested STRUCT with `city` and `state` (both STRING)
  - `children`: ARRAY of STRUCTs, each with `name` (STRING) and `age` (INT)
- **Type Mapping:** JSON types are mapped to Databricks SQL types such as STRING, INT, BOOLEAN, `STRUCT<>`, and `ARRAY<>`.
- **Example Schema:** `STRUCT<name: STRING, age: INT, address: STRUCT<city: STRING, state: STRING>, children: ARRAY<STRUCT<name: STRING, age: INT>>>`

## 6. Deriving and Applying the Schema
Converting a JSON string column into a STRUCT column takes two steps:
- **Step 1, schema_of_json:** Instead of defining the schema manually, pass an example JSON-formatted string to this built-in function, for example `SELECT schema_of_json('sample-json-string')`. It returns the inferred structure (schema) of the JSON string.
- **Step 2, from_json:** A Spark function that takes the JSON string and the schema from step 1 and returns a STRUCT column with the parsed data, for example `SELECT from_json(json_col, 'json-struct-schema') AS struct_column FROM table`.


---

# <b> Ingesting Enterprise Data Overview </b>
Lakeflow Connect Managed Connectors and Partner Connect streamline enterprise data ingestion by enabling fast and reliable integration from databases and applications into the Databricks Lakehouse through flexible, fully managed, and partner-supported options.

## 1. Why Managed Connectors
- **Context:** `CREATE TABLE`, `COPY INTO`, and Auto Loader cover ingestion from cloud object storage. Enterprise databases and SaaS applications need other techniques.
- **Two Options for Enterprise Sources:**
  - Lakeflow Connect Managed Connectors: built into Databricks.
  - Partner Connect: used when no native managed connector exists for the source.

## 2. Lakeflow Connect Managed Connectors
- **What They Are:** Connectors built into Databricks and designed to simplify ingesting data from a wide variety of enterprise databases and applications. They provide a low-code, fully managed experience, reducing the need for manual configuration or custom integration code.
- **Benefits:** Simplified ingestion from many sources, an easy-to-use user interface (or API), and infrastructure fully managed by Databricks.
- **Example Sources:** Workday, Salesforce, PostgreSQL, SQL Server, and more. The lecture diagram also shows Dynamics 365, Google Analytics, Oracle NetSuite, ServiceNow, and SharePoint.
- **Setup:** Point-and-click UI or API.
- **Release States:** Managed connectors are in various release states (some in public preview, others GA). Check the official documentation for the latest status.

## 3. SaaS vs Database Ingestion at a Glance

| Aspect | SaaS Connectors | Database Connectors |
|---|---|---|
| Sources | Publicly accessible sources such as APIs or OLAP endpoints (for example Salesforce, Workday) | External databases, on-prem or in a private cloud |
| Compute | Serverless Declarative Pipelines (managed ingestion pipeline) | Ingestion gateway on classic compute, then a serverless Declarative Pipelines job |
| Staging | None. Data goes straight to the target | State and staging data saved in a Unity Catalog volume |
| Credentials | Collected from Unity Catalog | Collected from Unity Catalog |
| Target | Streaming Delta Tables | Streaming Delta Tables |
| Complexity | Simpler | More involved (networking and load on the source database) |

## 4. SaaS Ingestion Architecture
- **Pipeline Type:** A managed ingestion pipeline connects to public SaaS sources (such as Salesforce or Workday), extracts the data, and ingests it directly into a streaming table. These pipelines are largely predefined and managed by Databricks, which handles source-specific complexities. They are set up through the UI or the API.
- **Step 1:** A Lakeflow serverless Declarative Pipelines job collects credentials from Unity Catalog.
- **Step 2:** The job reaches out to the publicly accessible data source (for example an API or an open OLAP port).
- **Step 3:** The service transforms the data and stores it in a Streaming Delta Table.
- **Efficiency:** Managed connectors use efficient incremental reads and writes, making ingestion faster, more scalable, and more cost-efficient while the data stays fresh for downstream consumption.
- **Control Plane vs Data Plane:** All data movement happens in the data plane. The control plane is only used for pipeline setup, monitoring (for example reading event logs), and management.

## 5. Database Ingestion Architecture
- **Why It Differs:** It is more involved than SaaS because it must connect to databases that may be on-prem or in a private cloud. The destination is the same: Streaming Delta Tables.
- **Step 1:** A classic compute Declarative Pipelines job (the ingestion gateway) collects credentials from Unity Catalog.
- **Step 2:** It uses the credentials to connect to the database sources and collect data.
- **Step 3:** The latest state and staging data are saved to a Unity Catalog volume.
- **Step 4:** A serverless Declarative Pipelines job processes the collected data into Streaming Delta Tables.
- **Two New Components:**
  - Ingestion Gateway: a dedicated pipeline that connects to the database to extract metadata, snapshots, and change logs, and stages all of it in the Unity Catalog volume.
  - Unity Catalog Volume: the intermediate staging layer that lets the next pipeline pick up and stream the data. It is secured with standard Unity Catalog mechanics, and by default access is limited to the user running the pipeline.
- **Why Add the Gateway:**
  - Networking: many databases sit behind a firewall or are not publicly accessible. Without an option like Private Link, the gateway can be deployed inside your network.
  - Load control: it limits direct connections to the source database. One gateway talks to the database and fans out to N pipelines for scalability. (This is not a problem for SaaS connectors, where load is controlled by API limits.)

## 6. Partner Connect
- **Use When:** No managed connector is available for your specific data source.
- **What It Does:** Lets you create trial accounts with select Databricks technology partners and connect your workspace to partner solutions directly from the Databricks UI. You can test solutions with your own data in the Lakehouse, then adopt the ones that best meet your business needs.
- **Ecosystem:** Partners offer a wide range of connectors with deep functionality maintained by dedicated teams of experienced engineers. Examples shown: Informatica, Qlik, Rivery, Alteryx, Prophecy, and Fivetran. They cover on-prem databases, cloud databases, file sources, and enterprise applications.
- **Customer Choice:** Databricks keeps working with partners to offer as many high-quality connectors as possible, even when a native connector exists. Native managed connectors are being added for the critical sources customers have requested.


---

# <b> Additional Features & Ingesting into Existing Delta Tables </b>
*Course: Data Ingestion with Lakeflow Connect (Data Engineer Learning Plan)*

## Features outside Lakeflow Connect managed connectors
- **Lakehouse Federation**: query external systems (e.g. MySQL, PostgreSQL, SQL Server, Snowflake, BigQuery) in place, without moving data. Best for ad hoc reporting, proofs of concept, exploring new ETL, and bridging an incremental migration.
- **Zerobus** (shown as "coming soon"): a Lakeflow Connect API to write event data directly into the lakehouse. About 100 MB/s throughput with under 5 s latency. Targets IoT, clickstream, telemetry.
- **Delta Sharing**: secure data sharing across platforms, clouds and regions, including consumers outside Databricks (Power BI, Spark, pandas, Tableau).
- **Databricks Marketplace**: open exchange built on Delta Sharing for datasets, notebooks, dashboards, ML models and solution accelerators.
  - Getting started: (1) open Marketplace from the workspace sidebar, (2) search assets, starting with those provided by Databricks, (3) click "Get instant access".

## MERGE INTO
Applies updates, inserts and deletes from a source into an existing Delta table in one atomic operation. Supports schema enforcement and evolution. Ideal for SCD, incremental loads and CDC.

| Row situation | Allowed actions |
|---|---|
| Matched | UPDATE or DELETE |
| Not matched by target | INSERT |
| Not matched by source | UPDATE or DELETE |

**Lesson walkthrough:** the source carries a status per row (delete / update / new). One MERGE removes the "delete" row, updates the "update" row, and inserts the "new" row. Target rows absent from the source (id=4) stay untouched.

**Syntax steps:** (1) declare target and source with aliases, (2) `ON` condition on a unique key, (3) `WHEN MATCHED AND ... THEN UPDATE SET`, (4) `WHEN MATCHED AND ... THEN DELETE`, (5) `WHEN NOT MATCHED THEN INSERT`.

## Extra exam notes (not from the lesson)
- `WHEN` clauses are evaluated in order; with several MATCHED clauses, every clause except the last needs a condition.
- If several source rows match one target row, MERGE errors, so deduplicate the source by key first.
```
MERGE INTO target_table AS t
USING source_table AS s
  ON t.id = s.id
WHEN MATCHED AND s.status = 'update' THEN
  UPDATE SET 
    t.email = s.email, 
    t.status = s.status
WHEN MATCHED AND s.status = 'delete' THEN
  DELETE
WHEN NOT MATCHED THEN
  INSERT (id, users, email, status) VALUES (s.id, s.users, s.email, s.status)
;
```